# 05 — Embeddings, residual blocks, and the full GPT-2 small (124,439,808 parameters)

The pieces that remain:

1. the token embedding: forward is indexing, backward is a scatter-add (and why `np.add.at` is required);
2. learned positional embeddings and their broadcast-sum gradient;
3. residual connections: the "gradient highway";
4. the pre-LN transformer block (and the bugs in your `TransformerBlock`);
5. the full GPT-2 small with **weight tying** and its double gradient contribution;
6. instantiating the real model and counting its parameters, component by component (124,439,808);
7. a gradient check of **every parameter** of a tiny model with the same architecture, then a **spot check of the full-size GPT-2 small**;
8. the initial loss of the real model on real text;
9. saving everything to `gpt_numpy.py` for the training notebook.

Memory: the full-size spot check runs GPT-2 small in float64 with gradients, which needs about 3 GB of RAM.

In [ ]:
import gc, math
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=120)

def numerical_grad(f, x, eps=1e-6):
    grad = np.zeros_like(x, dtype=np.float64)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad

def rel_error(a, b):
    return np.linalg.norm(a - b) / max(1e-12, np.linalg.norm(a) + np.linalg.norm(b))

def spot_check(f, p, g, k=20, eps=1e-5, seed=123):
    """Check k random entries of the analytic gradient g (of scalar f() w.r.t. array p) against
    central differences. Entries with near-zero gradient are skipped: there the finite difference
    is pure rounding noise, so a relative error would be meaningless."""
    r = np.random.default_rng(seed)
    candidates = np.flatnonzero(np.abs(g) > 1e-3 * np.abs(g).max())
    worst = 0.0
    for i in r.choice(candidates, size=min(k, candidates.size), replace=False):
        ii = np.unravel_index(i, p.shape)
        old = p[ii]
        p[ii] = old + eps; fp = f()
        p[ii] = old - eps; fm = f()
        p[ii] = old
        num = (fp - fm) / (2 * eps)
        worst = max(worst, abs(num - g[ii]) / (abs(num) + abs(g[ii])))
    return worst

rng = np.random.default_rng(0)

The layers from notebooks 01–04, unchanged (collapse this cell once you've read it):

In [ ]:
class Linear:
    def __init__(self, in_f, out_f, bias=True, std=0.02, rng=None, dtype=np.float64):
        rng = rng if rng is not None else np.random.default_rng()
        self.params = {"W": rng.normal(0.0, std, (in_f, out_f)).astype(dtype)}
        if bias:
            self.params["b"] = np.zeros(out_f, dtype=dtype)
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.x = None

    def forward(self, x):
        self.x = x
        out = x @ self.params["W"]
        if "b" in self.params:
            out = out + self.params["b"]
        return out

    def backward(self, dout):
        x2 = self.x.reshape(-1, self.x.shape[-1])
        d2 = dout.reshape(-1, dout.shape[-1])
        self.grads["W"] += x2.T @ d2
        if "b" in self.params:
            self.grads["b"] += d2.sum(axis=0)
        return dout @ self.params["W"].T

    def parameters(self, prefix=""):
        return [(prefix + k, self.params[k], self.grads[k]) for k in self.params]


class GELU:
    C = math.sqrt(2.0 / math.pi)        # Python float: keeps float32 arrays float32

    def forward(self, x):
        self.x = x
        self.t = np.tanh(self.C * (x + 0.044715 * x * x * x))
        return 0.5 * x * (1.0 + self.t)

    def backward(self, dout):
        x, t = self.x, self.t
        du = self.C * (1.0 + 3 * 0.044715 * x * x)
        return dout * (0.5 * (1.0 + t) + 0.5 * x * (1.0 - t**2) * du)

    def parameters(self, prefix=""):
        return []


class MLP:
    def __init__(self, C, n_layer, rng, dtype=np.float64):
        self.c_fc = Linear(C, 4 * C, rng=rng, dtype=dtype)
        self.gelu = GELU()
        self.c_proj = Linear(4 * C, C, std=0.02 / math.sqrt(2 * n_layer), rng=rng, dtype=dtype)

    def forward(self, x):
        return self.c_proj.forward(self.gelu.forward(self.c_fc.forward(x)))

    def backward(self, dout):
        return self.c_fc.backward(self.gelu.backward(self.c_proj.backward(dout)))

    def parameters(self, prefix=""):
        return self.c_fc.parameters(prefix + "c_fc.") + self.c_proj.parameters(prefix + "c_proj.")


class Softmax:
    def forward(self, x):
        z = x - x.max(axis=-1, keepdims=True)
        e = np.exp(z)
        self.s = e / e.sum(axis=-1, keepdims=True)
        return self.s

    def backward(self, dout):
        s = self.s
        return s * (dout - (dout * s).sum(axis=-1, keepdims=True))


class CrossEntropyLoss:
    def forward(self, logits, targets):
        B, T, V = logits.shape
        N = B * T
        l = logits.reshape(N, V)
        y = targets.reshape(N)
        m = l.max(axis=1, keepdims=True)
        lse = m + np.log(np.exp(l - m).sum(axis=1, keepdims=True))
        logp = l - lse
        self.probs = np.exp(logp)
        self.y = y
        self.shape = (B, T, V)
        return -logp[np.arange(N), y].mean()

    def backward(self):
        N = self.y.shape[0]
        d = self.probs.copy()
        d[np.arange(N), self.y] -= 1.0
        d /= N
        return d.reshape(self.shape)


class LayerNorm:
    def __init__(self, C, eps=1e-5, dtype=np.float64):
        self.eps = eps
        self.params = {"g": np.ones(C, dtype=dtype), "b": np.zeros(C, dtype=dtype)}
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, x):
        mu = x.mean(axis=-1, keepdims=True)
        xc = x - mu
        var = (xc**2).mean(axis=-1, keepdims=True)
        self.rstd = 1.0 / np.sqrt(var + self.eps)
        self.xhat = xc * self.rstd
        return self.xhat * self.params["g"] + self.params["b"]

    def backward(self, dout):
        C = dout.shape[-1]
        self.grads["g"] += (dout * self.xhat).reshape(-1, C).sum(axis=0)
        self.grads["b"] += dout.reshape(-1, C).sum(axis=0)
        h = dout * self.params["g"]
        return self.rstd * (h - h.mean(axis=-1, keepdims=True)
                            - self.xhat * (h * self.xhat).mean(axis=-1, keepdims=True))

    def parameters(self, prefix=""):
        return [(prefix + k, self.params[k], self.grads[k]) for k in self.params]


class CausalSelfAttention:
    def __init__(self, C, n_head, n_layer, rng, dtype=np.float64):
        assert C % n_head == 0
        self.C, self.H = C, n_head
        self.c_attn = Linear(C, 3 * C, rng=rng, dtype=dtype)
        self.c_proj = Linear(C, C, std=0.02 / math.sqrt(2 * n_layer), rng=rng, dtype=dtype)
        self.softmax = Softmax()

    def forward(self, x):
        B, T, C = x.shape
        H, hs = self.H, C // self.H
        qkv = self.c_attn.forward(x)
        q, k, v = np.split(qkv, 3, axis=-1)
        q = q.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        self.scale = 1.0 / math.sqrt(hs)    # Python float (np.sqrt would give float64 and upcast)
        att = (q @ k.transpose(0, 1, 3, 2)) * self.scale
        mask = np.triu(np.ones((T, T), dtype=bool), k=1)
        att = np.where(mask, -np.inf, att)
        A = self.softmax.forward(att)
        y = A @ v
        y = y.transpose(0, 2, 1, 3).reshape(B, T, C)
        self.q, self.k, self.v, self.A = q, k, v, A
        return self.c_proj.forward(y)

    def backward(self, dout):
        B, T, C = dout.shape
        H, hs = self.H, C // self.H
        dy = self.c_proj.backward(dout)
        dy = dy.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        dA = dy @ self.v.transpose(0, 1, 3, 2)
        dv = self.A.transpose(0, 1, 3, 2) @ dy
        datt = self.softmax.backward(dA) * self.scale
        dq = datt @ self.k
        dk = datt.transpose(0, 1, 3, 2) @ self.q
        merge = lambda t: t.transpose(0, 2, 1, 3).reshape(B, T, C)
        dqkv = np.concatenate([merge(dq), merge(dk), merge(dv)], axis=-1)
        return self.c_attn.backward(dqkv)

    def parameters(self, prefix=""):
        return self.c_attn.parameters(prefix + "c_attn.") + self.c_proj.parameters(prefix + "c_proj.")

## 1. Token embedding

**Forward:** `out = W[idx]`, with `W` of shape `(50257, 768)`, `idx` of shape `(B, T)` (token ids from tiktoken), and `out` of shape `(B, T, 768)`. Each token id selects its row: the id 31373 (`'hello'`) selects row 31373.

`wte` has 50,257 × 768 = **38,597,376 parameters, 31% of GPT-2 small.**

Indexing is the same as multiplying a one-hot matrix by $W$: $\text{out}_{bt} = \text{onehot}(idx_{bt})^\top W$. Using the Linear rule $dW = X^\top dY$ with $X$ = one-hot gives

$$
dW_v = \sum_{(b,t)\,:\,idx_{bt}=v} d\text{out}_{bt}
$$

So **each vocabulary row receives the sum of the gradients at every position where that token appeared.** Rows of tokens that didn't appear in the batch get zero gradient this step. There is no gradient for `idx` itself, since integers aren't differentiable, so the backward pass returns nothing.

### Why `np.add.at` and not `dW[idx] += dout`

Tokens repeat within a batch (think of how often "the" or a space appears). NumPy's fancy-index assignment `dW[idx] += dout` is **buffered**: for a repeated index only *one* of the additions survives. `np.add.at` is unbuffered and adds every occurrence.

In [ ]:
V, C = 5, 3
idx = np.array([[1, 2, 1, 1]])                    # token 1 appears 3 times
dout = np.ones((1, 4, C))

wrong = np.zeros((V, C)); wrong[idx] += dout
right = np.zeros((V, C)); np.add.at(right, idx, dout)
onehot = np.eye(V)[idx.ravel()]                   # (B*T, V)
reference = onehot.T @ dout.reshape(-1, C)        # the "Linear with one-hot input" view
print("dW[idx] += dout  -> row 1:", wrong[1], "  (WRONG: counted once)")
print("np.add.at        -> row 1:", right[1], "  (counted 3 times)")
print("matches one-hot matmul:", np.allclose(right, reference))

In [ ]:
class Embedding:
    def __init__(self, num, C, std=0.02, rng=None, dtype=np.float64):
        rng = rng if rng is not None else np.random.default_rng()
        self.params = {"W": rng.normal(0.0, std, (num, C)).astype(dtype)}
        self.grads = {"W": np.zeros_like(self.params["W"])}

    def forward(self, idx):
        self.idx = idx
        return self.params["W"][idx]

    def backward(self, dout):
        np.add.at(self.grads["W"], self.idx, dout)     # scatter-add; returns nothing (idx is not differentiable)

    def parameters(self, prefix=""):
        return [(prefix + "W", self.params["W"], self.grads["W"])]

emb = Embedding(V, C, std=1.0, rng=rng)
idx = rng.integers(0, V, size=(2, 6))
G = rng.normal(size=(2, 6, C))
f = lambda: np.sum(emb.forward(idx) * G)
f(); emb.backward(G)
print("embedding dW rel err:", rel_error(emb.grads["W"], numerical_grad(f, emb.params["W"])))

## 2. Positional embedding (learned, as in GPT-2)

GPT-2 does **not** use the sinusoidal encodings from the original Transformer paper. It learns a table `wpe` of shape `(1024, 768)` (786,432 parameters), exactly like the token table, indexed by position `0..T-1`:

```python
pos = wpe[np.arange(T)]          # (T, C)
x   = wte[idx] + pos             # (B, T, C) + (T, C)  -> broadcast over B
```

**Backward:** the addition copies `dx` to both terms (rule 3). The `(T, C)` position term was broadcast over B, so its gradient sums over B (rule 2): `dpos = dx.sum(axis=0)`. That `(T, 768)` gradient then goes through the same scatter-add as above, with `idx = arange(T)`, so no position repeats. Rows `T..1023` get no gradient when T < 1024.

(If you used *sinusoidal* encodings, as your `PositionalEncodings` stub might, they have no parameters, so there would be nothing to update and the gradient would simply stop there.)

## 3. Residual connections: the gradient highway

$$
x_{\text{out}} = x + F(x)\quad\Longrightarrow\quad dx = dx_\text{out} + \Big(\frac{\partial F}{\partial x}\Big)^{\!\top} dx_\text{out}
$$

In code: `dx = dout + F.backward(dout)`. The first term passes the gradient through **untouched**. Stack GPT-2 small's 12 blocks (24 residual sub-layers) and the gradient reaching the bottom is $dx_\text{top}$ *plus* correction terms. It can't vanish just because some sub-layer has small Jacobians. That's what makes training 12, 48 or 100+ layers possible. Without residuals, the gradient is a product of $L$ Jacobians, which tends to shrink or explode exponentially with depth.

In [ ]:
# demo: push a unit gradient back through GPT-2 small's 24 sub-layers (12 blocks x attn + MLP),
# each with a random 768x768 Jacobian J that shrinks vectors ~5x
C_demo = 768
g = rng.normal(size=C_demo); g /= np.linalg.norm(g)
g_plain, g_resid = g.copy(), g.copy()
for _ in range(24):
    J = rng.normal(0, 0.2 / np.sqrt(C_demo), (C_demo, C_demo))
    g_plain = J.T @ g_plain                  # y = F(x)      -> dx = J^T dy
    g_resid = g_resid + J.T @ g_resid        # y = x + F(x)  -> dx = dy + J^T dy
print(f"|grad| after 24 sub-layers   no residual: {np.linalg.norm(g_plain):.2e}   with residual: {np.linalg.norm(g_resid):.2f}")

## 4. The transformer block (pre-LN, GPT-2)

$$
\begin{aligned}
x_1 &= x_0 + \text{Attn}(\text{LN}_1(x_0))\\
x_2 &= x_1 + \text{MLP}(\text{LN}_2(x_1))
\end{aligned}
$$

**Pre-LN** (GPT-2) normalizes the *input* of each sub-layer and leaves the residual stream itself un-normalized, so the highway above is clean. **Post-LN** (original Transformer, 2017) is $x_1 = \text{LN}(x_0 + \text{Attn}(x_0))$. There every gradient must pass through a LayerNorm at each block, which is why post-LN needs careful learning-rate warmup and is harder to train deep.

**Backward**, in reverse order. Start with the second sub-block:

$$
dx_1 = dx_2 + \text{LN}_2.\text{backward}\big(\text{MLP}.\text{backward}(dx_2)\big),\qquad
dx_0 = dx_1 + \text{LN}_1.\text{backward}\big(\text{Attn}.\text{backward}(dx_1)\big)
$$

### Bugs in your `TransformerBlock.forward`

```python
out = inputs * self.multi_head_attention.forward(...)   # post: * must be +, the residual is an ADDITION
out = out * self.ffn.forward(out)                       # post: same
out = input + self.multi_head_attention.forward(...)    # pre : `input` is Python's built-in function -> use `inputs`
```

The multiplicative versions would also break the gradient highway: $d(x\odot F(x))$ has no clean pass-through term.

In [ ]:
class Block:
    def __init__(self, C, n_head, n_layer, rng, dtype=np.float64):
        self.ln_1 = LayerNorm(C, dtype=dtype)
        self.attn = CausalSelfAttention(C, n_head, n_layer, rng, dtype=dtype)
        self.ln_2 = LayerNorm(C, dtype=dtype)
        self.mlp = MLP(C, n_layer, rng, dtype=dtype)

    def forward(self, x):
        x = x + self.attn.forward(self.ln_1.forward(x))
        x = x + self.mlp.forward(self.ln_2.forward(x))
        return x

    def backward(self, dout):
        dx = dout + self.ln_2.backward(self.mlp.backward(dout))     # through x2 = x1 + mlp(ln2(x1))
        dx = dx + self.ln_1.backward(self.attn.backward(dx))        # through x1 = x0 + attn(ln1(x0))
        return dx

    def parameters(self, prefix=""):
        return (self.ln_1.parameters(prefix + "ln_1.") + self.attn.parameters(prefix + "attn.")
                + self.ln_2.parameters(prefix + "ln_2.") + self.mlp.parameters(prefix + "mlp."))

blk = Block(C=6, n_head=2, n_layer=1, rng=rng)
for name, p, _ in blk.parameters():
    p[...] = p + rng.normal(size=p.shape) * 0.5          # perturb away from 'all LN gains = 1'
x = rng.normal(size=(2, 4, 6)); G = rng.normal(size=(2, 4, 6))
f = lambda: np.sum(blk.forward(x) * G)
f()
dx = blk.backward(G)
print("block dx rel err:", rel_error(dx, numerical_grad(f, x)))
print("worst param rel err:", max(rel_error(g, numerical_grad(f, p)) for _, p, g in blk.parameters()))

One GPT-2 small block at full size (C = 768, 12 heads, 7,087,872 parameters), spot-checked:

In [ ]:
blk768 = Block(768, n_head=12, n_layer=12, rng=rng)
print(f"block parameters: {sum(p.size for _, p, _ in blk768.parameters()):,}")
x = rng.normal(size=(1, 8, 768)); G = rng.normal(size=(1, 8, 768))
f = lambda: np.sum(blk768.forward(x) * G)
f()
dx = blk768.backward(G)
print("dx spot rel err:", spot_check(f, x, dx))
print("worst param spot rel err:", max(spot_check(f, p, g, k=5) for _, p, g in blk768.parameters()))
del blk768

One thing to notice: inside `Block.forward`, the input `x` is used **twice**, once in the skip path and once in `ln_1`. The backward pass handles that with `dout + ...`. That's rule 2 (fan-out ⇒ sum) again. Every residual connection is a fan-out.

## 5. The full GPT-2 small

```
idx (B,T) ──wte──► (B,T,C) ─┐
arange(T) ──wpe──► (T,C) ───┴─(+)─► x ─► Block × L ─► ln_f ─► x_f (B,T,C) ─► x_f @ wteᵀ ─► logits (B,T,V)
```

### Weight tying

The output layer ("LM head") maps a C-vector to V logits. GPT-2 doesn't learn a separate `(C, V)` matrix: it **reuses the token embedding**, `logits = x_f @ wte.T`. Why:

- It saves $V\cdot C$ = 50,257 × 768 = 38,597,376 parameters. Without tying, GPT-2 small would have 163,037,184 parameters instead of 124,439,808.
- Tokens that should appear in similar contexts (output) and tokens that mean similar things (input) share one representation.

**Gradient consequence (rule 2 again).** `wte` is used twice, so its gradient has two contributions that **add**:

1. from the head: `logits = x_f @ W.T` is a Linear with weight $W^\top$, so $d(W^\top) = x_f^\top\,d\text{logits}$, i.e. $dW = d\text{logits}^\top x_f$, of shape $(V, C)$. This contribution is dense: every row gets gradient, because every vocabulary entry has a logit at every position.
2. from the input lookup: the sparse scatter-add from section 1.

This is exactly why gradients are accumulated with `+=`.

### The backward pass, in order

1. `dlogits` comes from the loss: `(probs − onehot) / N`.
2. Head: `wte.grad += dlogits_flatᵀ @ x_f_flat` and `dx = dlogits @ W`.
3. `dx = ln_f.backward(dx)`.
4. `for block in reversed(blocks): dx = block.backward(dx)`.
5. `x = tok + pos`, so both get `dx`: `wte.backward(dx)` (scatter-add, second contribution) and `wpe.backward(dx.sum(axis=0))`.

In [ ]:
class GPT:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=12, n_head=12, n_embd=768,
                 seed=0, dtype=np.float64):                      # defaults = GPT-2 small
        rng = np.random.default_rng(seed)
        self.block_size = block_size
        self.wte = Embedding(vocab_size, n_embd, rng=rng, dtype=dtype)   # token embedding AND output head (tied)
        self.wpe = Embedding(block_size, n_embd, rng=rng, dtype=dtype)   # learned positions
        self.h = [Block(n_embd, n_head, n_layer, rng, dtype=dtype) for _ in range(n_layer)]
        self.ln_f = LayerNorm(n_embd, dtype=dtype)

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.block_size, f"sequence length {T} > block_size {self.block_size}"
        x = self.wte.forward(idx) + self.wpe.forward(np.arange(T))   # (B,T,C) + (T,C)
        for block in self.h:
            x = block.forward(x)
        x = self.ln_f.forward(x)
        self.x_final = x                                             # cache for the tied head
        return x @ self.wte.params["W"].T                            # (B, T, V)

    def backward(self, dlogits):
        W = self.wte.params["W"]
        V, C = W.shape
        self.wte.grads["W"] += dlogits.reshape(-1, V).T @ self.x_final.reshape(-1, C)  # head contribution
        dx = dlogits @ W                                             # (B, T, C)
        dx = self.ln_f.backward(dx)
        for block in reversed(self.h):
            dx = block.backward(dx)
        self.wte.backward(dx)                                        # input contribution (adds to the above)
        self.wpe.backward(dx.sum(axis=0))                            # broadcast over B -> sum over B

    def parameters(self):
        ps = self.wte.parameters("wte.") + self.wpe.parameters("wpe.")
        for i, block in enumerate(self.h):
            ps += block.parameters(f"h.{i}.")
        return ps + self.ln_f.parameters("ln_f.")

    def zero_grad(self):
        for _, _, g in self.parameters():
            g.fill(0.0)

## 6. GPT-2 small, instantiated: every parameter counted

Per block: `ln_1` $2C$ + `c_attn` $3C^2+3C$ + attn `c_proj` $C^2+C$ + `ln_2` $2C$ + `c_fc` $4C^2+4C$ + mlp `c_proj` $4C^2+C$ $= 12C^2 + 13C$.

$$
\#\text{params} = \underbrace{VC}_{\text{wte (tied)}} + \underbrace{1024\cdot C}_{\text{wpe}} + L\,(12C^2+13C) + \underbrace{2C}_{\text{ln\_f}}
= 38{,}597{,}376 + 786{,}432 + 12 \times 7{,}087{,}872 + 1{,}536 = 124{,}439{,}808
$$

Now build the actual model (float32, 0.5 GB) and count from the arrays themselves:

In [ ]:
model = GPT(dtype=np.float32)                     # defaults: V=50257, block_size=1024, L=12, H=12, C=768
params = model.parameters()
total = sum(p.size for _, p, _ in params)

groups = {"wte (token embedding, tied head)": 0, "wpe (positions)": 0, "attention (12 blocks)": 0,
          "MLP (12 blocks)": 0, "LayerNorms (25)": 0}
for name, p, _ in params:
    if name.startswith("wte"):     groups["wte (token embedding, tied head)"] += p.size
    elif name.startswith("wpe"):   groups["wpe (positions)"] += p.size
    elif ".attn." in name:         groups["attention (12 blocks)"] += p.size
    elif ".mlp." in name:          groups["MLP (12 blocks)"] += p.size
    else:                          groups["LayerNorms (25)"] += p.size
for k, v in groups.items():
    print(f"{k:34}{v:>13,}  {100 * v / total:5.1f}%")
print(f"{'TOTAL':34}{total:>13,}")
print(f"\n{len(params)} parameter tensors, {total * 4 / 1e6:.0f} MB in float32")
for name, p, _ in params[:14]:
    print(f"  {name:24} {p.shape}")
print("  ... (same 12 tensors for h.1 ... h.11) ...")
for name, p, _ in params[-2:]:
    print(f"  {name:24} {p.shape}")

## 7. Gradient checks of the whole model

### 7.1 Every parameter of a tiny GPT with the same architecture

Checking every gradient needs two forward passes per parameter, so we use a 2-layer, 8-wide model (1,912 parameters). The code path is identical to GPT-2 small; only the numbers in the constructor differ. We scale the weights up from 0.02 so attention, LayerNorm and GELU all operate in non-trivial regimes.

In [ ]:
tiny = GPT(vocab_size=7, block_size=5, n_layer=2, n_head=2, n_embd=8, seed=3)
for name, p, _ in tiny.parameters():
    if name.endswith("W"):
        p *= 10
idx = rng.integers(0, 7, size=(2, 4))
targets = rng.integers(0, 7, size=(2, 4))
loss_fn = CrossEntropyLoss()
f = lambda: loss_fn.forward(tiny.forward(idx), targets)

tiny.zero_grad()
f()
tiny.backward(loss_fn.backward())
worst = 0.0
for name, p, g in tiny.parameters():
    err = rel_error(g, numerical_grad(f, p))
    worst = max(worst, err)
    print(f"{name:24} rel err {err:.2e}")
print("\nWORST:", worst)

### 7.2 Spot check of the real GPT-2 small

Now the full model, in float64, on real GPT-2 tokens. For every tensor of the embeddings, the first, middle and last blocks, and `ln_f`, we compare a few randomly chosen gradient entries with finite differences of the actual loss. Each check costs two full forward passes of GPT-2 small, so this takes a minute or two and about 3 GB of RAM. Set `RUN_FULL_SIZE_CHECK = False` to skip it.

Expected precision: single entries deep inside a 12-layer model typically agree to ~1e-7–1e-6 (the loss is ~11, and its last digits limit the finite difference). A bug would give ~1e-1 or worse.

In [ ]:
RUN_FULL_SIZE_CHECK = True

del model, params; gc.collect()                   # free the float32 model first
if RUN_FULL_SIZE_CHECK:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    toks = enc.encode("The quick brown fox jumps over the lazy dog because it wanted to see the other side.")
    idx, targets = np.array([toks[:6]]), np.array([toks[1:7]])   # (1, 6) real tokens
    model = GPT(dtype=np.float64)
    f = lambda: loss_fn.forward(model.forward(idx), targets)
    model.zero_grad()
    print("loss:", f())
    model.backward(loss_fn.backward())
    worst = 0.0
    for name, p, g in model.parameters():
        if name.startswith("h.") and name.split(".")[1] not in ("0", "5", "11"):
            continue
        err = spot_check(f, p, g, k=3)
        worst = max(worst, err)
        print(f"{name:24} {str(p.shape):14} spot rel err {err:.1e}")
    print("\nWORST:", worst)
    del model; gc.collect()

## 8. The initial loss of GPT-2 small on real text

From notebook 02: with GPT-2's initialization, expect $\ln 50257 + 768\cdot 0.02^2/2 \approx 10.98$.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
text = ("Language models are trained to predict the next token. Each position in a sequence sees only "
        "the tokens before it, and the model learns from every position at once. ") * 4
toks = np.array(enc.encode(text))
x, y = toks[None, :64], toks[None, 1:65]               # (1, 64) real tokens
model = GPT(dtype=np.float32)
print(f"initial loss {loss_fn.forward(model.forward(x), y):.3f}   expected ~ {np.log(50257) + 768 * 0.02**2 / 2:.3f}")
del model; gc.collect()

## 9. Save the model code for notebook 06

The cell below writes everything (layers, embeddings, block, GPT, loss) into `gpt_numpy.py` next to this notebook. It's the same code as above, collected into one file.

In [ ]:
%%writefile gpt_numpy.py
"""GPT-2 small in pure NumPy: every layer with a hand-written backward pass,
plus a loader for OpenAI's pretrained weights. Built in notebooks 01-05; notebook 06 imports it."""
import math
import numpy as np


# ---------------------------------------------------------------- helpers
def numerical_grad(f, x, eps=1e-6):
    """Central-difference gradient of scalar f() w.r.t. array x (perturbed in place)."""
    grad = np.zeros_like(x, dtype=np.float64)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        grad[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return grad


def rel_error(a, b):
    return np.linalg.norm(a - b) / max(1e-12, np.linalg.norm(a) + np.linalg.norm(b))


# ---------------------------------------------------------------- layers
class Linear:
    def __init__(self, in_f, out_f, bias=True, std=0.02, rng=None, dtype=np.float64):
        rng = rng if rng is not None else np.random.default_rng()
        self.params = {"W": rng.normal(0.0, std, (in_f, out_f)).astype(dtype)}
        if bias:
            self.params["b"] = np.zeros(out_f, dtype=dtype)
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.x = None

    def forward(self, x):
        self.x = x
        out = x @ self.params["W"]
        if "b" in self.params:
            out = out + self.params["b"]
        return out

    def backward(self, dout):
        x2 = self.x.reshape(-1, self.x.shape[-1])
        d2 = dout.reshape(-1, dout.shape[-1])
        self.grads["W"] += x2.T @ d2
        if "b" in self.params:
            self.grads["b"] += d2.sum(axis=0)
        return dout @ self.params["W"].T

    def parameters(self, prefix=""):
        return [(prefix + k, self.params[k], self.grads[k]) for k in self.params]


class GELU:
    C = math.sqrt(2.0 / math.pi)        # Python float: keeps float32 arrays float32

    def forward(self, x):
        self.x = x
        self.t = np.tanh(self.C * (x + 0.044715 * x * x * x))
        return 0.5 * x * (1.0 + self.t)

    def backward(self, dout):
        x, t = self.x, self.t
        du = self.C * (1.0 + 3 * 0.044715 * x * x)
        return dout * (0.5 * (1.0 + t) + 0.5 * x * (1.0 - t**2) * du)

    def parameters(self, prefix=""):
        return []


class MLP:
    def __init__(self, C, n_layer, rng, dtype=np.float64):
        self.c_fc = Linear(C, 4 * C, rng=rng, dtype=dtype)
        self.gelu = GELU()
        self.c_proj = Linear(4 * C, C, std=0.02 / math.sqrt(2 * n_layer), rng=rng, dtype=dtype)

    def forward(self, x):
        return self.c_proj.forward(self.gelu.forward(self.c_fc.forward(x)))

    def backward(self, dout):
        return self.c_fc.backward(self.gelu.backward(self.c_proj.backward(dout)))

    def parameters(self, prefix=""):
        return self.c_fc.parameters(prefix + "c_fc.") + self.c_proj.parameters(prefix + "c_proj.")


class Softmax:
    def forward(self, x):
        z = x - x.max(axis=-1, keepdims=True)
        e = np.exp(z)
        self.s = e / e.sum(axis=-1, keepdims=True)
        return self.s

    def backward(self, dout):
        s = self.s
        return s * (dout - (dout * s).sum(axis=-1, keepdims=True))


class CrossEntropyLoss:
    def forward(self, logits, targets):
        B, T, V = logits.shape
        N = B * T
        l = logits.reshape(N, V)
        y = targets.reshape(N)
        m = l.max(axis=1, keepdims=True)
        lse = m + np.log(np.exp(l - m).sum(axis=1, keepdims=True))
        logp = l - lse
        self.probs = np.exp(logp)
        self.y = y
        self.shape = (B, T, V)
        return -logp[np.arange(N), y].mean()

    def backward(self):
        N = self.y.shape[0]
        d = self.probs.copy()
        d[np.arange(N), self.y] -= 1.0
        d /= N
        return d.reshape(self.shape)


class LayerNorm:
    def __init__(self, C, eps=1e-5, dtype=np.float64):
        self.eps = eps
        self.params = {"g": np.ones(C, dtype=dtype), "b": np.zeros(C, dtype=dtype)}
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, x):
        mu = x.mean(axis=-1, keepdims=True)
        xc = x - mu
        var = (xc**2).mean(axis=-1, keepdims=True)
        self.rstd = 1.0 / np.sqrt(var + self.eps)
        self.xhat = xc * self.rstd
        return self.xhat * self.params["g"] + self.params["b"]

    def backward(self, dout):
        C = dout.shape[-1]
        self.grads["g"] += (dout * self.xhat).reshape(-1, C).sum(axis=0)
        self.grads["b"] += dout.reshape(-1, C).sum(axis=0)
        h = dout * self.params["g"]
        return self.rstd * (h - h.mean(axis=-1, keepdims=True)
                            - self.xhat * (h * self.xhat).mean(axis=-1, keepdims=True))

    def parameters(self, prefix=""):
        return [(prefix + k, self.params[k], self.grads[k]) for k in self.params]


class CausalSelfAttention:
    def __init__(self, C, n_head, n_layer, rng, dtype=np.float64):
        assert C % n_head == 0
        self.C, self.H = C, n_head
        self.c_attn = Linear(C, 3 * C, rng=rng, dtype=dtype)
        self.c_proj = Linear(C, C, std=0.02 / math.sqrt(2 * n_layer), rng=rng, dtype=dtype)
        self.softmax = Softmax()

    def forward(self, x):
        B, T, C = x.shape
        H, hs = self.H, C // self.H
        qkv = self.c_attn.forward(x)
        q, k, v = np.split(qkv, 3, axis=-1)
        q = q.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        self.scale = 1.0 / math.sqrt(hs)    # Python float (np.sqrt would give float64 and upcast)
        att = (q @ k.transpose(0, 1, 3, 2)) * self.scale
        mask = np.triu(np.ones((T, T), dtype=bool), k=1)
        att = np.where(mask, -np.inf, att)
        A = self.softmax.forward(att)
        y = A @ v
        y = y.transpose(0, 2, 1, 3).reshape(B, T, C)
        self.q, self.k, self.v, self.A = q, k, v, A
        return self.c_proj.forward(y)

    def backward(self, dout):
        B, T, C = dout.shape
        H, hs = self.H, C // self.H
        dy = self.c_proj.backward(dout)
        dy = dy.reshape(B, T, H, hs).transpose(0, 2, 1, 3)
        dA = dy @ self.v.transpose(0, 1, 3, 2)
        dv = self.A.transpose(0, 1, 3, 2) @ dy
        datt = self.softmax.backward(dA) * self.scale
        dq = datt @ self.k
        dk = datt.transpose(0, 1, 3, 2) @ self.q
        merge = lambda t: t.transpose(0, 2, 1, 3).reshape(B, T, C)
        dqkv = np.concatenate([merge(dq), merge(dk), merge(dv)], axis=-1)
        return self.c_attn.backward(dqkv)

    def parameters(self, prefix=""):
        return self.c_attn.parameters(prefix + "c_attn.") + self.c_proj.parameters(prefix + "c_proj.")


class Embedding:
    def __init__(self, num, C, std=0.02, rng=None, dtype=np.float64):
        rng = rng if rng is not None else np.random.default_rng()
        self.params = {"W": rng.normal(0.0, std, (num, C)).astype(dtype)}
        self.grads = {"W": np.zeros_like(self.params["W"])}

    def forward(self, idx):
        self.idx = idx
        return self.params["W"][idx]

    def backward(self, dout):
        np.add.at(self.grads["W"], self.idx, dout)

    def parameters(self, prefix=""):
        return [(prefix + "W", self.params["W"], self.grads["W"])]


class Block:
    def __init__(self, C, n_head, n_layer, rng, dtype=np.float64):
        self.ln_1 = LayerNorm(C, dtype=dtype)
        self.attn = CausalSelfAttention(C, n_head, n_layer, rng, dtype=dtype)
        self.ln_2 = LayerNorm(C, dtype=dtype)
        self.mlp = MLP(C, n_layer, rng, dtype=dtype)

    def forward(self, x):
        x = x + self.attn.forward(self.ln_1.forward(x))
        x = x + self.mlp.forward(self.ln_2.forward(x))
        return x

    def backward(self, dout):
        dx = dout + self.ln_2.backward(self.mlp.backward(dout))
        dx = dx + self.ln_1.backward(self.attn.backward(dx))
        return dx

    def parameters(self, prefix=""):
        return (self.ln_1.parameters(prefix + "ln_1.") + self.attn.parameters(prefix + "attn.")
                + self.ln_2.parameters(prefix + "ln_2.") + self.mlp.parameters(prefix + "mlp."))


class GPT:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=12, n_head=12, n_embd=768,
                 seed=0, dtype=np.float64):
        rng = np.random.default_rng(seed)
        self.block_size = block_size
        self.wte = Embedding(vocab_size, n_embd, rng=rng, dtype=dtype)
        self.wpe = Embedding(block_size, n_embd, rng=rng, dtype=dtype)
        self.h = [Block(n_embd, n_head, n_layer, rng, dtype=dtype) for _ in range(n_layer)]
        self.ln_f = LayerNorm(n_embd, dtype=dtype)

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.block_size
        x = self.wte.forward(idx) + self.wpe.forward(np.arange(T))
        for block in self.h:
            x = block.forward(x)
        x = self.ln_f.forward(x)
        self.x_final = x
        return x @ self.wte.params["W"].T

    def backward(self, dlogits):
        W = self.wte.params["W"]
        V, C = W.shape
        self.wte.grads["W"] += dlogits.reshape(-1, V).T @ self.x_final.reshape(-1, C)
        dx = dlogits @ W
        dx = self.ln_f.backward(dx)
        for block in reversed(self.h):
            dx = block.backward(dx)
        self.wte.backward(dx)
        self.wpe.backward(dx.sum(axis=0))

    def parameters(self):
        ps = self.wte.parameters("wte.") + self.wpe.parameters("wpe.")
        for i, block in enumerate(self.h):
            ps += block.parameters(f"h.{i}.")
        return ps + self.ln_f.parameters("ln_f.")

    def zero_grad(self):
        for _, _, g in self.parameters():
            g.fill(0.0)


# ---------------------------------------------------------------- pretrained weights
def load_safetensors(path):
    """Minimal .safetensors reader using only NumPy.
    File layout: 8-byte little-endian header length | JSON header | raw tensor bytes."""
    import json
    with open(path, "rb") as f:
        n = int.from_bytes(f.read(8), "little")
        header = json.loads(f.read(n))
    raw = np.memmap(path, dtype=np.uint8, mode="r")
    dtypes = {"F32": np.float32, "F16": np.float16, "F64": np.float64}
    out = {}
    for name, info in header.items():
        if name == "__metadata__":
            continue
        if info["dtype"] not in dtypes:
            raise ValueError(f"{name}: unsupported dtype {info['dtype']}")
        start, end = info["data_offsets"]
        out[name] = raw[8 + n + start: 8 + n + end].view(dtypes[info["dtype"]]).reshape(info["shape"])
    return out


def _hf_name_to_ours(k):
    """'h.3.attn.c_attn.weight' -> 'h.3.attn.c_attn.W', 'h.3.ln_1.weight' -> 'h.3.ln_1.g', etc."""
    if k.startswith("transformer."):
        k = k[len("transformer."):]
    parts = k.split(".")
    if k == "lm_head.weight" or parts[-2:] in (["attn", "bias"], ["attn", "masked_bias"]):
        return None                      # tied head / causal-mask buffers: not parameters
    stem, kind = ".".join(parts[:-1]), parts[-1]
    if kind == "weight":
        return stem + (".g" if parts[-2].startswith("ln_") else ".W")
    if kind == "bias":
        return stem + ".b"
    raise KeyError(k)


def load_hf_gpt2(model, path):
    """Copy HuggingFace GPT-2 weights into our model IN PLACE. Their Conv1D weights are stored
    (in, out), exactly like our x @ W, so no transposes are needed."""
    sd = load_safetensors(path)
    ours = {name: p for name, p, _ in model.parameters()}
    loaded = set()
    for k, arr in sd.items():
        name = _hf_name_to_ours(k)
        if name is None:
            continue
        p = ours[name]
        assert p.shape == arr.shape, f"{k}: file {arr.shape} vs model {p.shape}"
        p[...] = arr
        loaded.add(name)
    missing = set(ours) - loaded
    assert not missing, f"not found in file: {sorted(missing)[:5]}"
    return len(loaded)

In [ ]:
import importlib, gpt_numpy
importlib.reload(gpt_numpy)
m = gpt_numpy.GPT(vocab_size=11, block_size=8, n_layer=2, n_head=2, n_embd=8)
print("module OK, params:", sum(p.size for _, p, _ in m.parameters()))